**Main goal:** finding the **best model** for a **problem** (credit scoring problem in this case) according to **both** **predictive** a **fairness** **metrics** (using `PyFairnessAI` tools).

A classic ML approach is used: outer-inner evaluation, outer: train-test split, inner: cross-validation and HPO.

- Data: German data

- One sensitive variable: Age

- Using Pipelines

- Base models: 
  - Logistic Regression
  - XGBoost

- Base processing
  - Imputer
  - Encoder
  - Scaler

- Fairness processing (individual and multi)
  - Pre: Reweighing
  - In:
  - Post:

- Fairness metric: Separation (avg odds error)

- Predictive metric: balanced accuracy


In [53]:
import numpy as np
import pandas as pd
import os
from aif360.sklearn.datasets import fetch_german

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import tensorflow as tf

from PyMachineLearning.preprocessing import Encoder, Scaler, Imputer, ColumnTransformerToPandas
from PyMachineLearning.models import LogisticRegressionThreshold

from PyFairnessAI.model_selection import RandomizedSearchCVFairness, combined_score
from PyFairnessAI.preprocessing import ReweighingMetaEstimator
from PyFairnessAI.inprocessing import (AdversarialDebiasingEstimator, 
                                       ExponentiatedGradientReductionEstimator, 
                                       GridSearchReductionEstimator, Moment)
from PyFairnessAI.postprocessing import CalibratedEqualizedOdds, RejectOptionClassifier, PostProcessingMeta

import time 

from itertools import chain

pd.set_option('display.max_colwidth', None)

In [54]:
import warnings
warnings.filterwarnings("ignore")

In [55]:
def get_key_preprocessing_grid(model, fairness_processors, preprocessing_grid):

    if any(x in model for x in fairness_processors['multi']): # Fairness multi processor involved in the model
        key1 = model + '__estimator' + '__estimator'
        key3 = None
        preprocessing_grid_ = preprocessing_grid['fairness_processor'].copy()
        if any(x in model for x in fairness_processors['pre_in']):
            key2 = model + '__estimator'
        elif any(x in model for x in fairness_processors['pre_post']):
            key2 = model + '__estimator' + '__postprocessor'
        elif any(x in model for x in fairness_processors['post_pre']):
            key2 = model + '__postprocessor'   
        elif any(x in model for x in fairness_processors['post_in']):
           key2 = model + '__estimator'
           key3 = model + '__postprocessor'   
    else:
        key2 = key3 = None
        if any(x in model for x in fairness_processors['pre'] + fairness_processors['in']): # Fairness pre or in processor involved in the model
            if model == 'adv_debiasing':
                key1 = model
            else:
                key1 = model + '__estimator' 
            preprocessing_grid_ = preprocessing_grid['fairness_processor'].copy()
        elif any(x in model for x in fairness_processors['post']): # Fairness post processor involved in the model
            key1 = model + '__estimator' 
            key2 = model + '__postprocessor'
            preprocessing_grid_ = preprocessing_grid['fairness_processor'].copy()       
        else: # No fairness processor involved
            key1 = model
            preprocessing_grid_ = preprocessing_grid['not_fairness_processor'].copy()
    
    return key1, key2, key3, preprocessing_grid_

In [56]:
def get_model_param_grid(model, key):

    if 'log_reg' in model:

        param_grid = {f'{key}__penalty': ['l1', 'l2'],
                      f'{key}__C':  [0.01, 0.1, 1, 10, 30, 50, 75, 100],
                      f'{key}__class_weight': ['balanced', None],
                      f'{key}__threshold': [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
                    }
    
    elif 'XGB' in model:

        param_grid = {f'{key}__max_depth': [10, 20, 30, 40, 50, 70, 100],
                      f'{key}__reg_lambda': np.arange(0, 1, 0.05),
                      f'{key}__n_estimators':  [30, 50, 70, 100],
                      f'{key}__eta': np.arange(0, 0.3, 0.02),
                      f'{key}__alpha': np.arange(0.2, 1, 0.01)
                    }

    elif 'LGB' in model:

        param_grid = {f'{key}__max_depth': np.arange(2, 50),
                      f'{key}__num_leaves':  np.arange(2, 50),
                      f'{key}__n_estimators': [30, 50, 70, 100, 120, 150],
                      f'{key}__learning_rate':  np.arange(0.001, 0.1, 0.003),
                      f'{key}__lambda_l1': np.arange(0.001, 1, 0.005),
                      f'{key}__lambda_l2': np.arange(0.001, 1, 0.005),
                      f'{key}__min_split_gain':  np.arange(0.001, 0.01, 0.001),
                      f'{key}__min_child_weight': np.arange(5, 50),
                      f'{key}__lambda_feature_fraction':  np.arange(0.1, 0.95, 0.05)                                                                                                           
                      }
    
    elif 'RF' in model:
        
        param_grid = {f'{key}__max_depth': np.arange(2, 15),
                      f'{key}__min_samples_leaf':  np.arange(2, 15),
                      f'{key}__min_samples_split':  np.arange(2, 15),
                      f'{key}__n_estimators': [30, 50, 70, 100, 120],
                      f'{key}__criterion': ['gini', 'entropy']                                                                                                 
                    }
        
    elif 'adv_debiasing' in model:
        
        param_grid = {f'{key}__adversary_loss_weight': np.arange(0.01, 1, 0.03),
                      f'{key}__num_epochs':  np.arange(10, 100),
                      f'{key}__batch_size':  np.arange(70, 200),
                      f'{key}__classifier_num_hidden_units': np.arange(70, 300),
                      f'{key}__debias': [True, False]   
                    }
        
    return param_grid


In [57]:
def get_processor_param_grid(processor, key):

    if processor == 'expGR':

        param_grid = {f'{key}__constraints': ['DemographicParity', 'EqualizedOdds', 
                                              'TruePositiveRateParity', 'ErrorRateParity'],
                      f'{key}__eps': np.arange(0.001, 0.1, 0.003),
                      f'{key}__max_iter': np.arange(20, 100, 5),
                      f'{key}__eta0': np.arange(0.1, 4, 0.2),
                      f'{key}__drop_prot_attr': [True, False],
                    }   
        
    elif processor == 'GSR':

        param_grid = {f'{key}__constraints': ['DemographicParity', 'EqualizedOdds', 
                                              'TruePositiveRateParity', 'ErrorRateParity'],
                    f'{key}__constraint_weight': np.arange(0.01, 1, 0.05),
                    f'{key}__grid_size': np.arange(5, 50, 5),
                    f'{key}__grid_limit': np.arange(1, 10),
                    f'{key}__loss': ['ZeroOne', 'Square', 'Absolute'],                                
                    f'{key}__drop_prot_attr': [True, False],
                    }  
    
    elif processor == 'CEO':

        param_grid = {f'{key}__cost_constraint': ['fpr', 'fnr', 'weighted']}

    elif processor == 'ROC':

        param_grid = {f'{key}__threshold': np.arange(0.05, 0.5, 0.03), # must be between 0-0.5
                      f'{key}__margin': np.arange(0.01, 0.05, 0.005) # must be between 0-0.05
                    }
        
    return param_grid

In [58]:
def get_pipeline_param_grid(model, fairness_processors, preprocessing_grid):

    key1, key2, key3, preprocessing_grid_ = get_key_preprocessing_grid(model, fairness_processors, preprocessing_grid)
    param_grid = preprocessing_grid_.copy()
    param_grid.update(get_model_param_grid(model, key=key1)) 
    
    if any(x in model for x in fairness_processors['multi']): # Multi processor involved

        if 'reweighing_' in model: 

            param_grid.update(get_processor_param_grid(processor=model.split('_')[-1], key=key2)) 
        
        elif '_reweighing' in model: 

            param_grid.update(get_processor_param_grid(processor=model.split('_')[-2], key=key2)) 

        else:

            param_grid.update(get_processor_param_grid(processor=model.split('_')[-1], key=key2))
            param_grid.update(get_processor_param_grid(processor=model.split('_')[-2], key=key3))

    else: # Not multi processor (no fairness processor or pre/in/post processor)
                
        if any(x in model for x in fairness_processors['in']): # in processor  
            if model != 'adv_debiasing':
                param_grid.update(get_processor_param_grid(processor=model.split('_')[-1], key=model))

        elif any(x in model for x in fairness_processors['post']): # post processor 

            param_grid.update(get_processor_param_grid(processor=model.split('_')[-1], key=key2))
    
    return param_grid

In [59]:
def n_iter(model):

    if 'XGB' in model:
        return 10
    elif 'RF' in model:
        return 15
    else:
        return 30

---

In [68]:
X, y = fetch_german(binary_age=False)
X.index = range(len(X))
response_favorable_label = 1 # 'good' before encoding
sens_variable = 'age' # name of sensitive variable
X[sens_variable] = X.apply(lambda row: 1 if row[sens_variable] >= 25 else 0, axis=1).astype('category')
sens_priv_group = 1 # >= 25 years

In [ ]:
quant_predictors = [col for col in X.columns if X.dtypes[col] != 'category']
cat_predictors = [col for col in X.columns if col not in quant_predictors]  
predictors = quant_predictors + cat_predictors # X.columns

In [69]:
X.head()

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,other_parties,residence_since,...,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,sex,marital_status
0,<0,6,critical/other existing credit,radio/tv,1169,no known savings,>=7,4,none,4,...,1,none,own,2,skilled,1,yes,yes,male,single
1,0<=X<200,48,existing paid,radio/tv,5951,<100,1<=X<4,2,none,2,...,0,none,own,1,skilled,1,none,yes,female,div/dep/mar
2,no checking,12,critical/other existing credit,education,2096,<100,4<=X<7,2,none,3,...,1,none,own,1,unskilled resident,2,none,yes,male,single
3,<0,42,existing paid,furniture/equipment,7882,<100,4<=X<7,2,guarantor,4,...,1,none,for free,1,skilled,2,none,yes,male,single
4,<0,24,delayed previously,new car,4870,<100,1<=X<4,3,none,4,...,1,none,for free,2,skilled,2,none,yes,male,single


In [70]:
X.shape

(1000, 21)

In [71]:
y.head()

sex     age  foreign_worker
male    67   yes               good
female  22   yes                bad
male    49   yes               good
        45   yes               good
        53   yes                bad
Name: credit-risk, dtype: category
Categories (2, object): ['bad' < 'good']

In [72]:
encoder = Encoder(method='ordinal')
y = pd.Series(encoder.fit_transform(y.to_numpy().reshape(-1, 1)).flatten())

In [73]:
# needed for fairness post-processing
X.index = X[sens_variable] 
y.index = X[sens_variable]  

In [74]:
X.head()

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,other_parties,residence_since,...,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,sex,marital_status
age,,,,,,,,,,,,,,,,,,,,,
1,<0,6,critical/other existing credit,radio/tv,1169,no known savings,>=7,4,none,4,...,1,none,own,2,skilled,1,yes,yes,male,single
0,0<=X<200,48,existing paid,radio/tv,5951,<100,1<=X<4,2,none,2,...,0,none,own,1,skilled,1,none,yes,female,div/dep/mar
1,no checking,12,critical/other existing credit,education,2096,<100,4<=X<7,2,none,3,...,1,none,own,1,unskilled resident,2,none,yes,male,single
1,<0,42,existing paid,furniture/equipment,7882,<100,4<=X<7,2,guarantor,4,...,1,none,for free,1,skilled,2,none,yes,male,single
1,<0,24,delayed previously,new car,4870,<100,1<=X<4,3,none,4,...,1,none,for free,2,skilled,2,none,yes,male,single


In [75]:
y

age
1    1.0
0    0.0
1    1.0
1    1.0
1    0.0
    ... 
1    1.0
1    1.0
1    1.0
0    0.0
1    1.0
Length: 1000, dtype: float64

In [43]:
random_state = 123

In [44]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.85, random_state=random_state, stratify=y)
inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_state)

In [45]:
X_train

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,other_parties,residence_since,...,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,sex,marital_status
age,,,,,,,,,,,,,,,,,,,,,
1,no checking,36,existing paid,furniture/equipment,10974,<100,unemployed,4,none,2,...,1,none,own,2,high qualif/self emp/mgmt,1,yes,yes,female,div/dep/mar
1,<0,45,no credits/all paid,business,11816,<100,>=7,2,none,4,...,1,none,rent,2,skilled,1,none,yes,male,single
1,no checking,18,critical/other existing credit,radio/tv,1149,>=1000,1<=X<4,4,none,3,...,1,none,own,2,skilled,1,none,yes,male,single
1,no checking,42,critical/other existing credit,furniture/equipment,4042,500<=X<1000,1<=X<4,4,none,4,...,1,none,own,2,skilled,1,yes,yes,male,single
1,no checking,24,existing paid,radio/tv,3621,100<=X<500,>=7,2,none,4,...,1,none,own,2,skilled,1,none,yes,male,single
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1,no checking,36,existing paid,radio/tv,2299,500<=X<1000,>=7,4,none,4,...,1,none,own,1,skilled,1,none,yes,male,single
1,>=200,18,existing paid,furniture/equipment,3049,<100,<1,1,none,1,...,1,stores,own,1,unskilled resident,1,none,yes,female,div/dep/mar
1,no checking,15,existing paid,radio/tv,1213,500<=X<1000,>=7,4,none,3,...,1,stores,own,1,skilled,1,yes,yes,male,single


In [46]:
y_train

age
1    0.0
1    0.0
1    1.0
1    1.0
1    0.0
    ... 
1    1.0
1    1.0
1    1.0
1    1.0
1    1.0
Length: 850, dtype: float64

In [47]:
# Set up TensorFlow session (required by AdversarialDebiasingEstimator)
tf_session = tf.compat.v1.Session
# disable_eager_execution is required as well by TensorFlow
tf.compat.v1.disable_eager_execution()
tf.random.set_seed(random_state)

In [48]:
models, pipelines, fairness_processors, preprocessing_grid = {}, {}, {}, {}

quant_pipeline = Pipeline([
    ('imputer', Imputer()),
    ('scaler', Scaler())
    ])

cat_pipeline = Pipeline([
    ('imputer', Imputer(method='simple_most_frequent')),
    ('encoder', Encoder(method='one-hot', drop='first'))
    ])

quant_cat_processing = ColumnTransformer(transformers=[('quant', quant_pipeline, quant_predictors),
                                                       ('cat', cat_pipeline, cat_predictors)])

ceo = CalibratedEqualizedOdds(prot_attr=sens_variable, random_state=random_state) # Fairnes post-processor
roc = RejectOptionClassifier(prot_attr=sens_variable) # Fairnes post-processor

models['log_reg'] = LogisticRegressionThreshold(solver='liblinear', random_state=random_state)
models['XGB'] = XGBClassifier(random_state=random_state)
models['LGB'] = LGBMClassifier(random_state=random_state)
models['RF'] = RandomForestClassifier(random_state=random_state)

models['log_reg_reweighing'] = ReweighingMetaEstimator(estimator=models['log_reg'], prot_attr=sens_variable) # Fairness Pre-Processor
models['XGB_reweighing'] = ReweighingMetaEstimator(estimator=models['XGB'], prot_attr=sens_variable) # Fairness Pre-Processor
models['LGB_reweighing'] = ReweighingMetaEstimator(estimator=models['LGB'], prot_attr=sens_variable) # Fairness Pre-Processor
models['RF_reweighing'] = ReweighingMetaEstimator(estimator=models['RF'], prot_attr=sens_variable) # Fairness Pre-Processor

models['adv_debiasing'] = AdversarialDebiasingEstimator(prot_attr=sens_variable, scope_name='classifier', random_state=random_state) # Fairness In-Processor

models['log_reg_expGR'] = ExponentiatedGradientReductionEstimator(prot_attr=sens_variable, estimator=models['log_reg']) # Fairness In-Processor
models['XGB_expGR'] = ExponentiatedGradientReductionEstimator(prot_attr=sens_variable, estimator=models['XGB']) # Fairness In-Processor
models['LGB_expGR'] = ExponentiatedGradientReductionEstimator(prot_attr=sens_variable, estimator=models['LGB']) # Fairness In-Processor
models['RF_expGR'] = ExponentiatedGradientReductionEstimator(prot_attr=sens_variable, estimator=models['RF']) # Fairness In-Processor
models['log_reg_GSR'] = GridSearchReductionEstimator(prot_attr=sens_variable, estimator=models['log_reg']) # Fairness In-Processor
models['XGB_GSR'] = GridSearchReductionEstimator(prot_attr=sens_variable, estimator=models['XGB']) # Fairness In-Processor
models['LGB_GSR'] = GridSearchReductionEstimator(prot_attr=sens_variable, estimator=models['LGB']) # Fairness In-Processor
models['RF_GSR'] = GridSearchReductionEstimator(prot_attr=sens_variable, estimator=models['RF']) # Fairness In-Processor

models['log_reg_CEO'] = PostProcessingMeta(estimator=models['log_reg'], postprocessor=ceo, prefit=False, val_size=0.25) # Fairnes post-processor
models['XGB_CEO'] = PostProcessingMeta(estimator=models['XGB'], postprocessor=ceo, prefit=False, val_size=0.25) # Fairnes post-processor
models['LGB_CEO'] = PostProcessingMeta(estimator=models['LGB'], postprocessor=ceo, prefit=False, val_size=0.25) # Fairnes post-processor
models['RF_CEO'] = PostProcessingMeta(estimator=models['RF'], postprocessor=ceo, prefit=False, val_size=0.25) # Fairnes post-processor

models['log_reg_ROC'] = PostProcessingMeta(estimator=models['log_reg'], postprocessor=roc, prefit=False, val_size=0.25) # Fairnes post-processor
models['XGB_ROC'] = PostProcessingMeta(estimator=models['XGB'], postprocessor=roc, prefit=False, val_size=0.25) # Fairnes post-processor
models['LGB_ROC'] = PostProcessingMeta(estimator=models['LGB'], postprocessor=roc, prefit=False, val_size=0.25) # Fairnes post-processor
models['RF_ROC'] = PostProcessingMeta(estimator=models['RF'], postprocessor=roc, prefit=False, val_size=0.25) # Fairnes post-processor

models['log_reg_reweighing_expGR'] = ReweighingMetaEstimator(estimator=models['log_reg_expGR'], prot_attr=sens_variable) # pre + in
models['XGB_reweighing_expGR'] = ReweighingMetaEstimator(estimator=models['XGB_expGR'], prot_attr=sens_variable) # pre + in
models['LGB_reweighing_expGR'] = ReweighingMetaEstimator(estimator=models['LGB_expGR'], prot_attr=sens_variable) # pre + in
models['RF_reweighing_expGR'] = ReweighingMetaEstimator(estimator=models['RF_expGR'], prot_attr=sens_variable) # pre + in

models['log_reg_reweighing_GSR'] = ReweighingMetaEstimator(estimator=models['log_reg_GSR'], prot_attr=sens_variable) # pre + in
models['XGB_reweighing_GSR'] = ReweighingMetaEstimator(estimator=models['XGB_GSR'], prot_attr=sens_variable) # pre + in
models['LGB_reweighing_GSR'] = ReweighingMetaEstimator(estimator=models['LGB_GSR'], prot_attr=sens_variable) # pre + in
models['RF_reweighing_GSR'] = ReweighingMetaEstimator(estimator=models['RF_GSR'], prot_attr=sens_variable) # pre + in

models['log_reg_reweighing_CEO'] = ReweighingMetaEstimator(estimator=models['log_reg_CEO'], prot_attr=sens_variable) # pre + post
models['XGB_reweighing_CEO'] = ReweighingMetaEstimator(estimator=models['XGB_CEO'], prot_attr=sens_variable) # pre + post
models['LGB_reweighing_CEO'] = ReweighingMetaEstimator(estimator=models['LGB_CEO'], prot_attr=sens_variable) # pre + post
models['RF_reweighing_CEO'] = ReweighingMetaEstimator(estimator=models['RF_CEO'], prot_attr=sens_variable) # pre + post

models['log_reg_reweighing_ROC'] = ReweighingMetaEstimator(estimator=models['log_reg_ROC'], prot_attr=sens_variable) # pre + post
models['XGB_reweighing_ROC'] = ReweighingMetaEstimator(estimator=models['XGB_ROC'], prot_attr=sens_variable) # pre + post
models['LGB_reweighing_ROC'] = ReweighingMetaEstimator(estimator=models['LGB_ROC'], prot_attr=sens_variable) # pre + post
models['RF_reweighing_ROC'] = ReweighingMetaEstimator(estimator=models['RF_ROC'], prot_attr=sens_variable) # pre + post

models['log_reg_CEO_reweighing'] = PostProcessingMeta(estimator=models['log_reg_reweighing'], postprocessor=ceo, prefit=False, val_size=0.25)  # post + pre
models['XGB_CEO_reweighing'] = PostProcessingMeta(estimator=models['XGB_reweighing'], postprocessor=ceo, prefit=False, val_size=0.25)  # post + pre
models['LGB_CEO_reweighing'] = PostProcessingMeta(estimator=models['LGB_reweighing'], postprocessor=ceo, prefit=False, val_size=0.25)  # post + pre
models['RF_CEO_reweighing'] = PostProcessingMeta(estimator=models['RF_reweighing'], postprocessor=ceo, prefit=False, val_size=0.25)  # post + pre

models['log_reg_ROC_reweighing'] = PostProcessingMeta(estimator=models['log_reg_reweighing'], postprocessor=roc, prefit=False, val_size=0.25)  # post + pre
models['XGB_ROC_reweighing'] = PostProcessingMeta(estimator=models['XGB_reweighing'], postprocessor=roc, prefit=False, val_size=0.25)  # post + pre
models['LGB_ROC_reweighing'] = PostProcessingMeta(estimator=models['LGB_reweighing'], postprocessor=roc, prefit=False, val_size=0.25)  # post + pre
models['RF_ROC_reweighing'] = PostProcessingMeta(estimator=models['RF_reweighing'], postprocessor=roc, prefit=False, val_size=0.25)  # post + pre

models['log_reg_CEO_expGR'] = PostProcessingMeta(estimator=models['log_reg_expGR'], postprocessor=ceo, prefit=False, val_size=0.25)  # post + in
models['XGB_CEO_expGR'] = PostProcessingMeta(estimator=models['XGB_expGR'], postprocessor=ceo, prefit=False, val_size=0.25)  # post + in
models['LGB_CEO_expGR'] = PostProcessingMeta(estimator=models['LGB_expGR'], postprocessor=ceo, prefit=False, val_size=0.25)  # post + in
models['RF_CEO_expGR'] = PostProcessingMeta(estimator=models['RF_expGR'], postprocessor=ceo, prefit=False, val_size=0.25)  # post + in

models['log_reg_ROC_expGR'] = PostProcessingMeta(estimator=models['log_reg_expGR'], postprocessor=roc, prefit=False, val_size=0.25)  # post + in
models['XGB_ROC_expGR'] = PostProcessingMeta(estimator=models['XGB_expGR'], postprocessor=roc, prefit=False, val_size=0.25)  # post + in
models['LGB_ROC_expGR'] = PostProcessingMeta(estimator=models['LGB_expGR'], postprocessor=roc, prefit=False, val_size=0.25)  # post + in
models['RF_ROC_expGR'] = PostProcessingMeta(estimator=models['RF_expGR'], postprocessor=roc, prefit=False, val_size=0.25)  # post + in

models['log_reg_CEO_GSR'] = PostProcessingMeta(estimator=models['log_reg_GSR'], postprocessor=ceo, prefit=False, val_size=0.25)  # post + in
models['XGB_CEO_GSR'] = PostProcessingMeta(estimator=models['XGB_GSR'], postprocessor=ceo, prefit=False, val_size=0.25)  # post + in
models['LGB_CEO_GSR'] = PostProcessingMeta(estimator=models['LGB_GSR'], postprocessor=ceo, prefit=False, val_size=0.25)  # post + in
models['RF_CEO_GSR'] = PostProcessingMeta(estimator=models['RF_GSR'], postprocessor=ceo, prefit=False, val_size=0.25)  # post + in

models['log_reg_ROC_GSR'] = PostProcessingMeta(estimator=models['log_reg_GSR'], postprocessor=roc, prefit=False, val_size=0.25)  # post + in
models['XGB_ROC_GSR'] = PostProcessingMeta(estimator=models['XGB_GSR'], postprocessor=roc, prefit=False, val_size=0.25)  # post + in
models['LGB_ROC_GSR'] = PostProcessingMeta(estimator=models['LGB_GSR'], postprocessor=roc, prefit=False, val_size=0.25)  # post + in
models['RF_ROC_GSR'] = PostProcessingMeta(estimator=models['RF_GSR'], postprocessor=roc, prefit=False, val_size=0.25)  # post + in

fairness_processors['pre'] = ['reweighing']
fairness_processors['in'] = ['expGR', 'GSR', 'adv_debiasing'] 
fairness_processors['post'] = ['CEO', 'ROC']
fairness_processors['pre_in'] = ['reweighing_expGR', 'reweighing_GSR']
fairness_processors['pre_post'] = ['reweighing_CEO', 'reweighing_ROC']
fairness_processors['post_pre'] = ['CEO_reweighing', 'ROC_reweighing']
fairness_processors['post_in'] = ['CEO_expGR', 'ROC_expGR', 'CEO_GSR', 'ROC_GSR']
fairness_processors['multi'] = fairness_processors['pre_in'] + fairness_processors['pre_post'] + fairness_processors['post_pre'] + fairness_processors['post_in']
fairness_processors['all'] = list(chain(*[fairness_processors[x] for x in fairness_processors.keys()]))

for key, model in models.items():

    if  any(x in key for x in fairness_processors['all']): # model[key] involves fairness processor

        pipelines[key] = Pipeline([
                # Fairness processors need a Pandas df X as input to read the sens_variable_name
                ('preprocessing', ColumnTransformerToPandas(column_transformer=quant_cat_processing,
                                                            prot_attr=sens_variable,  
                                                            prot_attr_index=True)), # prot_attr_index=True needed for fairness post-processors
                (key, model) 
                ])            

    else:

        pipelines[key] = Pipeline([
                ('preprocessing', quant_cat_processing),
                (key, model) 
                ])
        
preprocessing_grid['not_fairness_processor'] = {'preprocessing__quant__scaler__apply': [True, False],
                                                'preprocessing__quant__scaler__method': ['standard', 'min-max'],
                                                'preprocessing__cat__encoder__method': ['ordinal', 'one-hot'],
                                                'preprocessing__cat__imputer__apply':  [False],
                                                'preprocessing__quant__imputer__apply': [False],
                                               #'preprocessing__cat__imputer__method':  ['simple_most_frequent'],
                                               #'preprocessing__quant__imputer__method': ['simple_mean', 'simple_median']
                                                }

# Same as  preprocessing_grid['not_fairness_preprocessor'] but adding '__column_transformer__' to the keys (needed in ColumnTransformerToPandas)        
preprocessing_grid['fairness_processor'] = {'__'.join([k.split('__')[0]] + ['column_transformer'] + k.split('__')[1:]) : v 
                                            for k, v in preprocessing_grid['not_fairness_processor'].items()}

In [ ]:
param_grid, best_results_list = {}, []

for model in pipelines.keys():

    print(model)

    param_grid[model] = get_pipeline_param_grid(model, fairness_processors, preprocessing_grid)

    fairness_random_search = RandomizedSearchCVFairness(estimator=pipelines[model], 
                                                        param_distributions=param_grid[model], 
                                                        fairness_scoring='average_odds_error', 
                                                        predictive_scoring='balanced_accuracy',
                                                        objective='combined', 
                                                        fairness_scoring_direction='minimize',
                                                        predictive_scoring_direction='maximize',
                                                        fairness_weight=0.5, predictive_weight=0.5,
                                                        cv=inner, n_iter=n_iter(model), random_state=random_state,
                                                        prot_attr=sens_variable, 
                                                        priv_group=sens_priv_group,
                                                        pos_label=response_favorable_label)

    start_time = time.time()
    fairness_random_search.fit(X=X_train, y=y_train)
    end_time = time.time()
    elapsed_time = np.round(end_time - start_time, 3)
    best_result_dict = {'model': model, 'time': elapsed_time}
    best_result_dict.update(dict(fairness_random_search.cv_results_.iloc[0]))
    best_results_list.append(best_result_dict)

In [51]:
best_results = pd.DataFrame(best_results_list)
best_results['combined-score'] = combined_score(predictive_scores=best_results['predictive-score'], 
                                                fairness_scores=best_results['fairness-score'], 
                                                predictive_scoring_direction='maximize', 
                                                fairness_scoring_direction='minimize',
                                                predictive_weight=0.5, fairness_weight=0.5)
best_results = best_results.sort_values(by='combined-score', ascending=False)

In [27]:
# Define the results path
results_path = r'C:\Users\fscielzo\Documents\IBiDat\Fairness AI\PyFairnessAI-package\notebooks\Project-notebooks\results\best_results_fairness_workflow.csv'

# Check if the file already exists
if not os.path.exists(results_path):
    # If the file doesn't exist, save the CSV
    best_results.to_csv(results_path)
    print(f"File saved at: {results_path}")
else:
    print(f"File already exists at: {results_path}")

Bottle necks (most expensive pipelines):

- XGB_expGR

- XGB_reweighing_expGR

- XGB_CEO_expGR

- XGB_ROC_expGR

- RF_expGR

- RF_reweighing_expGR

- RF_ROC_expGR

- RF_CEO_expGR

In [10]:
results_path = r'C:\Users\fscielzo\Documents\IBiDat\Fairness AI\PyFairnessAI-package\notebooks\Project-notebooks\results\best_results_fairness_workflow.csv'
best_results = pd.read_csv(results_path, index_col=0)

In [12]:
best_results.head()

,model,time,params,predictive-score,fairness-score,combined-score
9,log_reg_expGR,85.996,"{'preprocessing__column_transformer__quant__scaler__apply': True, 'preprocessing__column_transformer__quant__scaler__method': 'standard', 'preprocessing__column_transformer__cat__encoder__method': 'one-hot', 'preprocessing__column_transformer__cat__imputer__apply': False, 'preprocessing__column_transformer__quant__imputer__apply': False, 'log_reg_expGR__estimator__penalty': 'l1', 'log_reg_expGR__estimator__C': 30, 'log_reg_expGR__estimator__class_weight': None, 'log_reg_expGR__estimator__threshold': 0.7, 'log_reg_expGR__constraints': 'ErrorRateParity', 'log_reg_expGR__eps': 0.043000000000000003, 'log_reg_expGR__max_iter': 40, 'log_reg_expGR__eta0': 3.900000000000001, 'log_reg_expGR__drop_prot_attr': True}",0.706162,0.087089,0.835028
53,log_reg_ROC_expGR,57.514,"{'preprocessing__column_transformer__quant__scaler__apply': False, 'preprocessing__column_transformer__quant__scaler__method': 'standard', 'preprocessing__column_transformer__cat__encoder__method': 'one-hot', 'preprocessing__column_transformer__cat__imputer__apply': False, 'preprocessing__column_transformer__quant__imputer__apply': False, 'log_reg_ROC_expGR__estimator__estimator__penalty': 'l2', 'log_reg_ROC_expGR__estimator__estimator__C': 0.1, 'log_reg_ROC_expGR__estimator__estimator__class_weight': None, 'log_reg_ROC_expGR__estimator__estimator__threshold': 0.7, 'log_reg_ROC_expGR__estimator__constraints': 'EqualizedOdds', 'log_reg_ROC_expGR__estimator__eps': 0.085, 'log_reg_ROC_expGR__estimator__max_iter': 60, 'log_reg_ROC_expGR__estimator__eta0': 2.900000000000001, 'log_reg_ROC_expGR__estimator__drop_prot_attr': False, 'log_reg_ROC_expGR__postprocessor__threshold': 0.26, 'log_reg_ROC_expGR__postprocessor__margin': 0.045}",0.694118,0.082719,0.814251
25,log_reg_reweighing_expGR,85.850,"{'preprocessing__column_transformer__quant__scaler__apply': True, 'preprocessing__column_transformer__quant__scaler__method': 'standard', 'preprocessing__column_transformer__cat__encoder__method': 'one-hot', 'preprocessing__column_transformer__cat__imputer__apply': False, 'preprocessing__column_transformer__quant__imputer__apply': False, 'log_reg_reweighing_expGR__estimator__estimator__penalty': 'l1', 'log_reg_reweighing_expGR__estimator__estimator__C': 100, 'log_reg_reweighing_expGR__estimator__estimator__class_weight': 'balanced', 'log_reg_reweighing_expGR__estimator__estimator__threshold': 0.7, 'log_reg_reweighing_expGR__estimator__constraints': 'ErrorRateParity', 'log_reg_reweighing_expGR__estimator__eps': 0.085, 'log_reg_reweighing_expGR__estimator__max_iter': 65, 'log_reg_reweighing_expGR__estimator__eta0': 2.7000000000000006, 'log_reg_reweighing_expGR__estimator__drop_prot_attr': True}",0.700840,0.098181,0.804014
4,log_reg_reweighing,5.928,"{'preprocessing__column_transformer__quant__scaler__apply': True, 'preprocessing__column_transformer__quant__scaler__method': 'standard', 'preprocessing__column_transformer__cat__encoder__method': 'one-hot', 'preprocessing__column_transformer__cat__imputer__apply': False, 'preprocessing__column_transformer__quant__imputer__apply': False, 'log_reg_reweighing__estimator__penalty': 'l1', 'log_reg_reweighing__estimator__C': 1, 'log_reg_reweighing__estimator__class_weight': None, 'log_reg_reweighing__estimator__threshold': 0.9}",0.668347,0.053245,0.803538
29,log_reg_reweighing_GSR,61.872,"{'preprocessing__column_transformer__quant__scaler__apply': False, 'preprocessing__column_transformer__quant__scaler__method': 'standard', 'preprocessing__column_transformer__cat__encoder__method': 'one-hot', 'preprocessing__column_transformer__cat__imputer__apply': False, 'preprocessing__column_transformer__quant__imputer__apply': False, 'log_reg_reweighing_GSR__estimator__estimator__penalty': 'l2', 'log_reg_reweighing_GSR__estimator__estimator__C': 30, 'log_reg_reweighing_GSR__estimator__estimator__class_weight': None, 'log_reg_reweighing_GSR__estimator__estimator__thresho

In [8]:
best_results.shape

(65, 6)

In [9]:
best_results.tail()

,model,time,params,predictive-score,fairness-score,combined-score
47,LGB_ROC_reweighing,15.474,"{'preprocessing__column_transformer__quant__scaler__apply': False, 'preprocessing__column_transformer__quant__scaler__method': 'standard', 'preprocessing__column_transformer__cat__encoder__method': 'one-hot', 'preprocessing__column_transformer__cat__imputer__apply': False, 'preprocessing__column_transformer__quant__imputer__apply': False, 'LGB_ROC_reweighing__estimator__estimator__max_depth': 16, 'LGB_ROC_reweighing__estimator__estimator__num_leaves': 38, 'LGB_ROC_reweighing__estimator__estimator__n_estimators': 70, 'LGB_ROC_reweighing__estimator__estimator__learning_rate': 0.016, 'LGB_ROC_reweighing__estimator__estimator__lambda_l1': 0.34600000000000003, 'LGB_ROC_reweighing__estimator__estimator__lambda_l2': 0.646, 'LGB_ROC_reweighing__estimator__estimator__min_split_gain': 0.004, 'LGB_ROC_reweighing__estimator__estimator__min_child_weight': 18, 'LGB_ROC_reweighing__estimator__estimator__lambda_feature_fraction': 0.6000000000000002, 'LGB_ROC_reweighing__postprocessor__threshold': 0.29, 'LGB_ROC_reweighing__postprocessor__margin': 0.039999999999999994}",0.500000,0.000000,0.500000
20,RF_CEO,8.956,"{'preprocessing__column_transformer__quant__scaler__apply': True, 'preprocessing__column_transformer__quant__scaler__method': 'standard', 'preprocessing__column_transformer__cat__encoder__method': 'ordinal', 'preprocessing__column_transformer__cat__imputer__apply': False, 'preprocessing__column_transformer__quant__imputer__apply': False, 'RF_CEO__estimator__max_depth': 13, 'RF_CEO__estimator__min_samples_leaf': 5, 'RF_CEO__estimator__min_samples_split': 4, 'RF_CEO__estimator__n_estimators': 30, 'RF_CEO__estimator__criterion': 'entropy', 'RF_CEO__postprocessor__cost_constraint': 'fpr'}",0.596639,0.143395,0.485067
51,LGB_CEO_expGR,304.551,"{'preprocessing__column_transformer__quant__scaler__apply': False, 'preprocessing__column_transformer__quant__scaler__method': 'standard', 'preprocessing__column_transformer__cat__encoder__method': 'one-hot', 'preprocessing__column_transformer__cat__imputer__apply': False, 'preprocessing__column_transformer__quant__imputer__apply': False, 'LGB_CEO_expGR__estimator__estimator__max_depth': 3, 'LGB_CEO_expGR__estimator__estimator__num_leaves': 42, 'LGB_CEO_expGR__estimator__estimator__n_estimators': 150, 'LGB_CEO_expGR__estimator__estimator__learning_rate': 0.031, 'LGB_CEO_expGR__estimator__estimator__lambda_l1': 0.536, 'LGB_CEO_expGR__estimator__estimator__lambda_l2': 0.081, 'LGB_CEO_expGR__estimator__estimator__min_split_gain': 0.003, 'LGB_CEO_expGR__estimator__estimator__min_child_weight': 28, 'LGB_CEO_expGR__estimator__estimator__lambda_feature_fraction': 0.45000000000000007, 'LGB_CEO_expGR__estimator__constraints': 'DemographicParity', 'LGB_CEO_expGR__estimator__eps': 0.010000000000000002, 'LGB_CEO_expGR__estimator__max_iter': 80, 'LGB_CEO_expGR__estimator__eta0': 3.500000000000001, 'LGB_CEO_expGR__estimator__drop_prot_attr': False, 'LGB_CEO_expGR__postprocessor__cost_constraint': 'fpr'}",0.597759,0.151387,0.474282
18,XGB_CEO,5.210,"{'preprocessing__column_transformer__quant__scaler__apply': True, 'preprocessing__column_transformer__quant__scaler__method': 'min-max', 'preprocessing__column_transformer__cat__encoder__method': 'ordinal', 'preprocessing__column_transformer__cat__imputer__apply': False, 'preprocessing__column_transformer__quant__imputer__apply': False, 'XGB_CEO__estimator__max_depth': 70, 'XGB_CEO__estimator__reg_lambda': 0.7000000000000001, 'XGB_CEO__estimator__n_estimators': 30, 'XGB_CEO__estimator__eta': 0.28, 'XGB_CEO__estimator__alpha': 0.25000000000000006, 'XGB_CEO__postprocessor__cost_constraint': 'fpr'}",0.654622,0.281409,0.388961
50,XGB_CEO_expGR,525.608,"{'preprocessing__column_transformer__quant__scaler__apply': True, 'preprocessing__column_transformer__quant__scaler__method': 'min-max', 'preprocessing__column_transformer__cat__encoder__method': 'ordinal', 'preprocessing__column_tr

---

**Important observation regarding the combined score:**

- The current combined metric has a problem. If for a certain model the best (max) predictive metric is 0.50 and the best (min) fairness is 0.30,
in both cases the metric is bad, but if a hyper-param set has that combination of metrics (balanced_acc=0.5, average_odds_error=0.30) the combined score will be 1 (due to how it is defined by normalization), so, perfect, and this model will be considered the best overall compared with other models, even though its predictive and fairness metrics are bad individually.

    - Possible solution: when comparing different models being careful regarding selecting the "best" just based on the combined score, because the above behavior is a possibility. Instead, apply the combined score methodology to the final individual scores and compute a new combined score, which would be suitable for selecting the best model since this new approach guaranty that the best model will be the one with the best trade-off between predictive power and fairness.

    The idea is to use combined_score(predictive_scores=final_predictive_scores, fairness_scores=final_fairness_scores) where final_predictive_scores and final_fairness_scores are the *final* predictive and fairness scores of the models, obtained in the HPO phase (optimized on the combined score, that is, with objective='combined').

The idea is to apply the combined score in two steps/stages:

1) For optimizing the hyperparams of each model and obtaining their best version according to both predictive power and fairness

2) For selecting the final best model according to both predictive power and fairness